### RAG   
RAG (Retrieval-Augmented Generation) is a technique that retrieves relevant information from external data sources and provides it to an LLM, enabling more accurate and context-aware responses.

### RAG workflow:  
User Question   
      ↓    
Embedding   
      ↓   
FAISS Search    
      ↓     
Retrieve Relevant Documents   
      ↓    
Generate Answer

**Create Documents:**

In [2]:
documents = [
    "Artificial Intelligence is the simulation of human intelligence by machines.",
    "Machine Learning is a subset of Artificial Intelligence.",
    "Deep Learning uses neural networks with many layers.",
    "Python is one of the most popular programming languages.",
    "Natural Language Processing enables computers to understand human language."
]

**Generate Embeddings**

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(documents)

print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(5, 384)


**Create FAISS Index**

In [4]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

Now we have a vector database.

**Create query**

In [5]:
query = "What is Machine Learning?"

#query embedding
query_embedding = model.encode([query])

**Search Knowledge base**

In [6]:
distances, indices = index.search(
    np.array(query_embedding),
    k=2
)

print(indices)

[[1 0]]


**Retrieve Context**

In [7]:
for idx in indices[0]:
    print(documents[idx])

Machine Learning is a subset of Artificial Intelligence.
Artificial Intelligence is the simulation of human intelligence by machines.





The expacted output is the most relevant chunks

**Simulate the LLM**

Retrieved Context   
        ↓   
GPT / LLM   
        ↓   
Answer

**For now:**

In [8]:
retrieved_docs = [documents[i] for i in indices[0]]

answer = " ".join(retrieved_docs)

print(answer)

Machine Learning is a subset of Artificial Intelligence. Artificial Intelligence is the simulation of human intelligence by machines.


**Here is All Code**

In [13]:
embeddings = model.encode(documents)

print("sentences and dimensions",embeddings.shape)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

query = "What is Machine Learning?"

#query embedding
query_embedding = model.encode([query])

distances, indices = index.search(
    np.array(query_embedding),
    k=2
)

print("index numbers:",indices)

for idx in indices[0]:
    print(documents[idx])

print('------coonect sentences---------------')
#in paragraph type
retrieved_docs = [documents[i] for i in indices[0]]

answer = " ".join(retrieved_docs)

print(answer)    

sentences and dimensions (5, 384)
index numbers: [[1 0]]
Machine Learning is a subset of Artificial Intelligence.
Artificial Intelligence is the simulation of human intelligence by machines.
------coonect sentences---------------
Machine Learning is a subset of Artificial Intelligence. Artificial Intelligence is the simulation of human intelligence by machines.


**TRY WITH Multiple Queries**

In [18]:
# Step 1: Encode documents
embeddings = model.encode(documents)
print("Sentences and dimensions:", embeddings.shape)

# Step 2: Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("\n" + "=" * 50)
print("        FAISS SEMANTIC SEARCH RESULTS")
print("=" * 50)

# Step 3: Multiple queries in a loop
queries = [
    "Explain AI",
    "What is Deep Learning?",
    "Tell me about Python",
    "What is NLP?"
]

for query in queries:
    print(f"\nQuery: {query}")
    print("-" * 40)

    # Generate query embedding
    query_embedding = model.encode([query])

    # Search FAISS - top 2 results
    distances, indices = index.search(np.array(query_embedding), k=2)

    # Print top 2 results
    retrieved_docs = [documents[i] for i in indices[0]]
    for i, doc in enumerate(retrieved_docs):
        print(f"  Result {i+1}: {doc}")
        print(f"  Distance: {distances[0][i]:.4f}")

    # Joined paragraph answer
    answer = " ".join(retrieved_docs)
    print(f"\n  Combined Answer: {answer}")

print("\n" + "=" * 50)
print("Search complete!")

Sentences and dimensions: (5, 384)

        FAISS SEMANTIC SEARCH RESULTS

Query: Explain AI
----------------------------------------
  Result 1: Artificial Intelligence is the simulation of human intelligence by machines.
  Distance: 0.5859
  Result 2: Machine Learning is a subset of Artificial Intelligence.
  Distance: 0.9362

  Combined Answer: Artificial Intelligence is the simulation of human intelligence by machines. Machine Learning is a subset of Artificial Intelligence.

Query: What is Deep Learning?
----------------------------------------
  Result 1: Deep Learning uses neural networks with many layers.
  Distance: 0.6055
  Result 2: Machine Learning is a subset of Artificial Intelligence.
  Distance: 0.9194

  Combined Answer: Deep Learning uses neural networks with many layers. Machine Learning is a subset of Artificial Intelligence.

Query: Tell me about Python
----------------------------------------
  Result 1: Python is one of the most popular programming languages.
  D